# 19. 표준 라이브러리 산책

기초 · 문제 091–095

**사용법**
1. 바로 아래 **🔧 채점기 준비** 셀을 먼저 실행합니다(▶).
2. 각 문제의 `%%writefile` 셀에 코드를 작성하고 실행하면 파일로 저장됩니다.
3. 그 아래 `check(번호)` 셀을 실행하면 채점됩니다.
4. 직접 입력을 넣어 실행해 보려면 새 셀에 `%run p001.py`처럼 입력합니다.

> 정답과 해설은 위키독스 본문의 접힌 상자에 있습니다. 충분히 고민한 뒤에 펼쳐 보세요.

In [ ]:
#@title 🔧 채점기 준비 (먼저 실행하세요)
TESTS = {91: [['7\n3 1 3 2 1 5 1', '1 3', True, None], ['4\n9 8 9 8', '8 2', True, None], ['1\n-7', '-7 1', False, None], ['3\n5 6 7', '5 1', False, None], [None, '-45 1058', False, 'import random\nrandom.seed(11)\nn = 100000\nprint(n)\nprint(*[random.randint(-50, 50) for _ in range(n)])']], 92: [['5 21\n5 6 7 8 9', '21', True, None], ['4 15\n1 2 9 9', '12', True, None], ['3 100\n1 2 3', '6', False, None], ['6 500\n93 181 245 214 315 36', '495', False, None], [None, '149999', False, 'import random\nrandom.seed(12)\nn = 100\nm = 150000\nprint(n, m)\nprint(*[random.randint(1, 100000) for _ in range(n)])']], 93: [['2026-11-23\n2027-01-01', '월요일\n39', True, None], ['2000-01-01\n2000-01-01', '토요일\n0', False, None], ['2024-03-01\n2024-02-28', '금요일\n-2', False, None], ['1582-10-15\n2026-11-23', '금요일\n162207', False, None], ['9999-12-31\n0001-01-01', '금요일\n-3652058', False, None]], 94: [['10', '3', True, None], ['1000000000000000000000000000000000000', '1000000000000000000', True, None], ['0', '0', False, None], ['1', '1', False, None], ['99', '9', False, None], ['100', '10', False, None], ['10000000200000000', '100000000', False, None], ['1000000000000002000000000000000', '1000000000000000', False, None], ['1000000000000000002000000000000000000', '1000000000000000000', False, None], ['10000000000000000000000000000000000000000', '100000000000000000000', False, None], ['170141183460469231731687303715884105727', '13043817825332782212', False, None]], 95: [['5\n9 3 1 7 3\n3\n3\n0\n10', '3\n0\n5', True, None], ['1\n5\n2\n5\n4', '1\n0', False, None], [None, 'sha256:36633ffbfd8ecaf78de18445ef173fcf6d1bb69a947da159bfaab07b8b875cf8', False, 'import random\nrandom.seed(13)\nn = q = 100000\nprint(n)\nprint(*[random.randint(-10**9, 10**9) for _ in range(n)])\nprint(q)\nfor _ in range(q):\n    print(random.randint(-10**9, 10**9))']]}
TL = {91: 2, 92: 2, 93: 2, 94: 2, 95: 2}
import subprocess, sys, pathlib, hashlib

_GEN_CACHE = {}

def _norm(s):
    return "\n".join(l.rstrip() for l in s.strip("\n").splitlines()).strip("\n")

def _input_of(inp, gen):
    if gen is None:
        return inp
    if gen not in _GEN_CACHE:
        _GEN_CACHE[gen] = subprocess.run([sys.executable, "-c", gen], capture_output=True, text=True).stdout
    return _GEN_CACHE[gen]

def check(pid):
    """p{번호}.py 를 모든 테스트로 실행해 채점합니다."""
    path = pathlib.Path(f"p{pid:03d}.py")
    if not path.exists():
        print(f"먼저 위의 %%writefile p{pid:03d}.py 셀을 실행하세요."); return
    limit = TL[pid]
    ok = 0
    for i, (inp, exp, shown, gen) in enumerate(TESTS[pid], 1):
        tag = f"예제 {i}" if shown else f"숨은 테스트 {i}"
        data = _input_of(inp, gen)
        try:
            r = subprocess.run([sys.executable, str(path)], input=data.rstrip("\n") + "\n",
                               capture_output=True, text=True, timeout=limit)
        except subprocess.TimeoutExpired:
            print(f"⏱ {tag}: 시간 초과 ({limit}초)"); continue
        got = _norm(r.stdout)
        if exp.startswith("sha256:"):
            got_cmp = "sha256:" + hashlib.sha256(got.encode()).hexdigest()
        else:
            got_cmp = got
        if r.returncode:
            last = r.stderr.strip().splitlines()[-1] if r.stderr.strip() else "알 수 없는 오류"
            print(f"💥 {tag}: 실행 오류 → {last}")
        elif got_cmp == exp:
            ok += 1; print(f"✅ {tag}")
        elif shown:
            print(f"❌ {tag}\n   입력: {inp!r}\n   기대: {exp!r}\n   실제: {got!r}")
        elif gen is not None:
            print(f"❌ {tag}  (큰 입력)")
        else:
            print(f"❌ {tag}  (입력: {inp[:60]!r})")
    total = len(TESTS[pid])
    print("─" * 30)
    print(f"🎉 {pid:03d}번 통과! ({ok}/{total})" if ok == total else f"{ok}/{total} 통과 — 다시 도전해 보세요.")
print('채점기 준비 완료!')

## 091. Counter와 최빈값

첫째 줄에 N, 둘째 줄에 N개의 정수가 주어집니다. 가장 많이 나온 값과 그 횟수를 출력하십시오. 그런 값이 여럿이면 **가장 작은 값**을 출력합니다.

**제한** 1 ≤ N ≤ 10⁵, |각 수| ≤ 10⁹

**예제 1**
입력
```
7
3 1 3 2 1 5 1
```
출력
```
1 3
```

**예제 2**
입력
```
4
9 8 9 8
```
출력
```
8 2
```

In [ ]:
%%writefile p091.py
# 여기에 코드를 작성하세요


In [ ]:
check(91)

## 092. 스물하나

첫째 줄에 카드 수 N과 상한 M, 둘째 줄에 N장의 카드 숫자가 주어집니다. 서로 다른 카드 3장을 골라 합이 M을 넘지 않으면서 M에 가장 가깝게 만들 때 그 합을 출력하십시오. 그런 조합은 반드시 존재합니다.

**제한** 3 ≤ N ≤ 100, 1 ≤ 카드 숫자 ≤ 10⁵, 3 ≤ M ≤ 3×10⁵

**예제 1**
입력
```
5 21
5 6 7 8 9
```
출력
```
21
```

**예제 2**
입력
```
4 15
1 2 9 9
```
출력
```
12
```

In [ ]:
%%writefile p092.py
# 여기에 코드를 작성하세요


In [ ]:
check(92)

## 093. 며칠 남았을까

두 줄에 걸쳐 날짜가 `YYYY-MM-DD` 형식으로 주어집니다. 첫째 줄에 첫 날짜의 요일(`월요일`~`일요일`)을, 둘째 줄에 첫 날짜에서 둘째 날짜까지의 날수(둘째 − 첫째, 음수 가능)를 출력하십시오.

**제한** 연도는 1 이상 9999 이하

**예제 1**
입력
```
2026-11-23
2027-01-01
```
출력
```
월요일
39
```

In [ ]:
%%writefile p093.py
# 여기에 코드를 작성하세요


In [ ]:
check(93)

## 094. 정확한 제곱근

정수 n이 주어질 때 ⌊√n⌋(제곱해서 n을 넘지 않는 가장 큰 정수)를 출력하십시오.

**제한** 0 ≤ n ≤ 10⁴⁰

**예제 1**
입력
```
10
```
출력
```
3
```

**예제 2**
입력
```
1000000000000000000000000000000000000
```
출력
```
1000000000000000000
```

In [ ]:
%%writefile p094.py
# 여기에 코드를 작성하세요


In [ ]:
check(94)

## 095. bisect로 세기

첫째 줄에 N, 둘째 줄에 N개의 정수, 셋째 줄에 질의 수 Q, 이어서 Q줄에 정수 x가 주어집니다. 각 질의마다 N개의 수 가운데 **x 이하인 수의 개수**를 출력하십시오.

**제한** 1 ≤ N, Q ≤ 10⁵, |각 수| ≤ 10⁹

**예제 1**
입력
```
5
9 3 1 7 3
3
3
0
10
```
출력
```
3
0
5
```

In [ ]:
%%writefile p095.py
# 여기에 코드를 작성하세요


In [ ]:
check(95)